> **MIT License** - Copyright (c) 2026 QuantWall. Full text: https://opensource.org/license/mit
>
> **Disclaimer.** This content is general information and software-engineering education only. It is not financial product advice, a recommendation, or an offer to buy or sell any financial product, and it does not take into account your objectives, financial situation or needs. The author does not hold an Australian Financial Services Licence and is not a registered Commodity Trading Advisor. Trading futures and leveraged products carries a high risk of loss. Consult a licensed adviser before making any financial decision.
>
> **HYPOTHETICAL OR SIMULATED PERFORMANCE RESULTS HAVE CERTAIN LIMITATIONS. UNLIKE AN ACTUAL PERFORMANCE RECORD, SIMULATED RESULTS DO NOT REPRESENT ACTUAL TRADING. ALSO, SINCE THE TRADES HAVE NOT BEEN EXECUTED, THE RESULTS MAY HAVE UNDER-OR-OVER COMPENSATED FOR THE IMPACT, IF ANY, OF CERTAIN MARKET FACTORS, SUCH AS LACK OF LIQUIDITY. SIMULATED TRADING PROGRAMS IN GENERAL ARE ALSO SUBJECT TO THE FACT THAT THEY ARE DESIGNED WITH THE BENEFIT OF HINDSIGHT. NO REPRESENTATION IS BEING MADE THAT ANY ACCOUNT WILL OR IS LIKELY TO ACHIEVE PROFIT OR LOSSES SIMILAR TO THOSE SHOWN.**
>
> All data shown is synthetic or publicly available sample data. Features are standard textbook baselines (lagged returns, ATR, RSI, moving averages, volume). No proprietary signal is disclosed or implied.

# The map: every number in the video

One synthetic idea (the month-end idea: futures drift up in the last two trading days of each month while funds rebalance) taken through every stop. Each cell reproduces one number from the video with the same code the animations use (`animations_manim/map_sim.py`). Change the inputs and watch the decisions move. Requirements: `numpy`. Each rate uses 2,000 simulated worlds, so cells take a few seconds.

## The synthetic world
24 futures markets, one 2-day window per market per month, 1.7% volatility per window, average correlation 0.5, fat tails (a t-distribution with 4 degrees of freedom), changing volatility. The claim: +0.20% per window; assumed cost 0.04% per round trip (illustrative).

In [ ]:
import numpy as np

WINDOWS_PER_YEAR = 12
WINDOW_SD = 0.017          # one 2-day window: 1.7% standard deviation per market
RHO = 0.5                  # average correlation between markets
DF = 4                     # fat tails
GROSS = 0.0020             # the claim: +0.20% per window, before costs
COST_RT = 0.0004           # assumed cost: 0.02% a side, 0.04% a round trip (illustrative)
NET = GROSS - COST_RT      # +0.16% per window after costs
CRIT = 1.645               # one-sided 5% pass mark


def world(rng, mu, markets=24, years=20, rho=RHO):
    """Window returns (windows x markets): common factor + idiosyncratic, fat tails, changing volatility."""
    n = years * WINDOWS_PER_YEAR
    scale = np.sqrt((DF - 2) / DF)
    vol = np.exp(rng.normal(0, 0.25, n))
    vol /= np.sqrt((vol ** 2).mean())
    z = (np.sqrt(rho) * rng.standard_t(DF, n)[:, None] * scale
         + np.sqrt(1 - rho) * rng.standard_t(DF, (n, markets)) * scale)
    return mu + WINDOW_SD * vol[:, None] * z


def tstat(x):
    return float(x.mean() / (x.std(ddof=1) / np.sqrt(len(x))))


def pooled_t(r):
    """The mistake: every market-window counted as an independent observation."""
    return tstat(r.ravel())


def portfolio_t(r):
    """The fix: one number per window (the average across markets), so correlated markets count once."""
    return tstat(r.mean(axis=1))


def pass_rate(seed, mu, markets, years, test=portfolio_t, worlds=2000):
    rng = np.random.default_rng(seed)
    ts = np.array([test(world(rng, mu, markets, years)) for _ in range(worlds)])
    return float(np.mean(ts > CRIT)), ts


def hook():
    """Two tests, both wrong: a real edge on a small test fails; pure noise counted wrongly passes."""
    small, t_small = pass_rate(11, NET, 1, 5)
    wrong, t_wrong = pass_rate(12, 0.0, 24, 20, test=pooled_t)
    return small, t_small, wrong, t_wrong


def costs():
    """Cost check from the trade plan alone (no returns): break-even and a x3 stress."""
    trips = WINDOWS_PER_YEAR
    return {"trips_per_year": trips, "cost_per_year": trips * COST_RT, "net": NET, "net_x3": GROSS - 3 * COST_RT}


def power():
    """Chance the (correct) test passes if the claimed edge is real, by design."""
    designs = [(1, 5), (1, 20), (24, 5), (24, 20)]
    return {d: pass_rate(20 + k, NET, *d)[0] for k, d in enumerate(designs)}


def calibration():
    """The exact test, run on noise worlds with the data's quirks: how often does it pass?"""
    naive, _ = pass_rate(31, 0.0, 24, 20, test=pooled_t)
    fixed, _ = pass_rate(32, 0.0, 24, 20, test=portfolio_t)
    return naive, fixed


def the_look(seed=41):
    """One world where the claimed edge is real; the single run, net of costs."""
    r = world(np.random.default_rng(seed), NET)
    port = r.mean(axis=1)
    return {"t": portfolio_t(r), "t_x3": tstat(port - 2 * COST_RT), "share_up": float(np.mean(r.mean(axis=0) > 0)),
            "markets": r.shape[1], "curve": np.cumsum(port)}


def forward(seed=51, months=12, paths=4000, path_seed=55):
    """Expected range of a 12-month forward record if the edge is what the backtest says; one forward path.
    The shown path (seed 55) was picked because it dips inside the range: the case a kill rule is written for."""
    rng = np.random.default_rng(seed)
    sims = np.array([world(rng, NET, 24, 1).mean(axis=1) for _ in range(paths)])[:, :months]
    cum = np.cumsum(sims, axis=1)
    lo, mid, hi = (np.percentile(cum, q, axis=0) for q in (5, 50, 95))
    path = np.cumsum(world(np.random.default_rng(path_seed), NET, 24, 1).mean(axis=1))[:months]
    return lo, mid, hi, path

## Hook: two tests, both wrong

In [ ]:
small, _, wrong, _ = hook()
print(f"real edge, 1 market x 5 years: passes {small:.1%}  (fails about {1 - small:.0%})")
print(f"pure noise, 24 markets counted as independent: passes {wrong:.1%}  (promised 5%)")

## Stop 2: the cost check (no returns needed)

In [ ]:
c = costs()
print(f"{c['trips_per_year']} round trips a year: {c['cost_per_year']:.2%} a year per market")
print(f"per window: claim {GROSS:.2%}, after costs {c['net']:.2%}, with costs x3 {c['net_x3']:.2%}")

## Stop 3: power, by design
The chance the correct (portfolio) test passes if the claimed edge is real. Rule: no test below 50%.

In [ ]:
for (markets, years), p in power().items():
    print(f"{markets:2d} markets x {years:2d} years: power {p:.1%}")

## Stop 5: test the test on noise worlds

In [ ]:
naive, fixed = calibration()
print(f"pooled (every market-window independent): passes noise {naive:.1%}")
print(f"portfolio (one number per window):          passes noise {fixed:.1%}")

## Stop 7: the look (one world where the edge is real)

In [ ]:
L = the_look()
print(f"portfolio t = {L['t']:.2f} (pass mark {CRIT})")
print(f"average per window with costs x3: {(L['curve'][-1] / len(L['curve']) - 2 * COST_RT):+.3%}")

## Stop 8: the range a forward year can take
If the edge is exactly what the backtest says: 5th to 95th percentile of the cumulative 12-month record, and the path shown in the video (seed 55, picked because it dips inside the range).

In [ ]:
lo, mid, hi, path = forward()
print("month  5th pct  median  95th pct  path")
for m in range(12):
    print(f"{m + 1:5d}  {lo[m]:+7.1%}  {mid[m]:+6.1%}  {hi[m]:+8.1%}  {path[m]:+6.1%}")

### Try this
- Stop 3: drop the correlation `RHO` to 0.2. How much power do 24 markets add now?
- Stop 5: run the pooled test with `rho=0.0`. Is it still wrong?
- Stop 2: double `COST_RT`. Where does the idea die?

*Simulated results on synthetic data. Education only. Not financial advice.*